# MS-Dialog — Phase 1 Uncertainty Analysis
## Model: `gemini-2.5-flash` | Dataset: MS-Dialog (100 tech-support cases)

Covers **Single-Turn** (1 CQ round) and **Multi-Turn** (3 CQ rounds) experiments.

**Research question**: When a tech-support model is uncertain, does it ask an *EPISTEMIC* CQ
(reducible knowledge gap) or an *ALEATORIC* CQ (irreducible input ambiguity)?
Does the CQ type predict how much confidence is gained?

Sections:
1. Setup & Load Data
2. **Single-Turn Analysis** — confidence progression, CQ type, confidence Δ × CQ type, category breakdown
3. **Multi-Turn Analysis** — 4-checkpoint confidence arc, per-turn CQ type, CQ diversity
4. **Cross-Experiment Comparison**
5. Export Summary CSV

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../").resolve()))

DATASET  = "ms-dialog"
MODEL_ID = "gemini-2.5-flash"
ROOT     = Path("../../").resolve()
OUTPUTS  = ROOT / "outputs" / DATASET / MODEL_ID

import warnings, logging
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)
matplotlib.use("Agg")
sns.set_theme(style="whitegrid", palette="muted")
FIGSIZE = (9, 5)

print(f"Model:   {MODEL_ID}")
print(f"Outputs: {OUTPUTS}")
print(f"Exists:  {OUTPUTS.exists()}")

## 2. Load Data

In [ ]:
# ── Generation results ────────────────────────────────────────────────────────
st_results = pd.read_csv(OUTPUTS / "phase1_singleturn_results.csv")
mt_results = pd.read_csv(OUTPUTS / "phase1_multiturn_results.csv")

# ── Judge labels ──────────────────────────────────────────────────────────────
st_labels  = pd.read_csv(OUTPUTS / "phase1_singleturn_classified.csv")
mt_labels  = pd.read_csv(OUTPUTS / "phase1_multiturn_classified.csv")

# ── Drop blocked rows ─────────────────────────────────────────────────────────
st = st_results[~st_results["was_blocked"]].copy()
mt = mt_results[~mt_results["was_blocked"]].copy()

print(f"Single-turn : {len(st_results)} rows, {int(st_results['was_blocked'].sum())} blocked → {len(st)} usable")
print(f"Multi-turn  : {len(mt_results)} rows, {int(mt_results['was_blocked'].sum())} blocked → {len(mt)} usable")

# ── Resolve id field (dataset uses case_id) ───────────────────────────────────
for df in [st, mt, st_results, mt_results]:
    if "id" not in df.columns and "case_id" in df.columns:
        df["id"] = df["case_id"]

# ── Helpers ───────────────────────────────────────────────────────────────────
def pct(s): return s.mean() * 100

def jaccard(a, b):
    ta = set(str(a).lower().split())
    tb = set(str(b).lower().split())
    if not ta and not tb: return 1.0
    return len(ta & tb) / len(ta | tb)

print(f"\nSingle-turn CQs classified : {len(st_labels)}")
print(f"Multi-turn  CQs classified : {len(mt_labels)}")

---
## 3. Single-Turn Analysis
*One CQ round — model generates preliminary solution + CQ, user responds, model gives updated solution*

### 3.1 Confidence Progression (Overall)

In [ ]:
prelim_mean  = st["preliminary_confidence"].mean()
updated_mean = st["updated_confidence"].mean()
delta        = st["updated_confidence"] - st["preliminary_confidence"]

print("=== Single-Turn Confidence Progression ===")
print(f"  Preliminary (before CQ) : {prelim_mean:.1f}")
print(f"  Updated     (after CQ)  : {updated_mean:.1f}")
print(f"  Mean delta              : {delta.mean():+.1f}")
print(f"  Median delta            : {delta.median():+.1f}")
print(f"  Std                     : {delta.std():.1f}")
print(f"  Gained confidence       : {(delta > 0).sum()} cases")
print(f"  No change               : {(delta == 0).sum()} cases")
print(f"  Lost confidence         : {(delta < 0).sum()} cases")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar: prelim vs updated
axes[0].bar(["Preliminary", "Updated (post-CQ)"], [prelim_mean, updated_mean],
            color=["steelblue", "seagreen"], width=0.45)
axes[0].set_ylabel("Mean Confidence")
axes[0].set_ylim(0, 100)
axes[0].set_title(f"Confidence: Before vs After CQ")
for i, v in enumerate([prelim_mean, updated_mean]):
    axes[0].text(i, v + 1, f"{v:.1f}", ha="center", fontweight="bold")

# Histogram of delta
axes[1].hist(delta, bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", lw=1.5, linestyle="--", label="No change")
axes[1].axvline(delta.mean(), color="orange", lw=1.5, label=f"Mean={delta.mean():+.1f}")
axes[1].set_xlabel("Confidence delta (pp)")
axes[1].set_ylabel("Cases")
axes[1].set_title("Δ Confidence Distribution")
axes[1].legend(fontsize=9)

fig.suptitle(f"Single-Turn Confidence — {MODEL_ID} (n={len(st)})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_confidence.png", dpi=150)
plt.show()
print("Saved: st_fig_confidence.png")

### 3.2 CQ Type Distribution

In [ ]:
q_col_st = "clarifying_question" if "clarifying_question" in st_labels.columns else "question"
st_valid = st_labels[st_labels["label"].isin({"EPISTEMIC", "ALEATORIC"})].copy()

print(f"CQs classified: {len(st_valid)} / {len(st)}")
vc = st_valid["label"].value_counts()
print()
for lbl in ["EPISTEMIC", "ALEATORIC"]:
    n = vc.get(lbl, 0)
    print(f"  {lbl}: {n} ({100*n/len(st_valid):.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
colors = {"EPISTEMIC": "steelblue", "ALEATORIC": "salmon"}
ax.bar(vc.index, vc.values, color=[colors.get(l, "gray") for l in vc.index], width=0.45)
ax.set_ylabel("Count")
ax.set_title(f"CQ Type Distribution — {MODEL_ID} (single-turn)")
for i, (lbl, v) in enumerate(vc.items()):
    ax.text(i, v + 0.5, f"{v}\n({100*v/len(st_valid):.0f}%)", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_cq_type.png", dpi=150)
plt.show()
print("Saved: st_fig_cq_type.png")

### 3.3 Confidence Delta × CQ Type  ← *Core Research Result*
Does the type of CQ predict how much confidence is gained after the user responds?

In [ ]:
# Merge CQ type into single-turn results
st_with_type = st.merge(
    st_valid[[q_col_st, "label"]].rename(columns={q_col_st: "clarifying_question", "label": "cq_type"}),
    on="clarifying_question", how="left"
)
st_with_type["conf_delta"] = st_with_type["updated_confidence"] - st_with_type["preliminary_confidence"]

typed = st_with_type[st_with_type["cq_type"].isin({"EPISTEMIC", "ALEATORIC"})]

print("=== Confidence Delta by CQ Type ===")
print()
grp = typed.groupby("cq_type")["conf_delta"].agg(["mean", "median", "std", "count"]).round(2)
display(grp.rename(columns={"mean": "Mean Δ", "median": "Median Δ", "std": "Std", "count": "n"}))

# Statistical test: Mann-Whitney U (non-parametric, no normality assumption)
from scipy.stats import mannwhitneyu
ep_vals = typed[typed["cq_type"] == "EPISTEMIC"]["conf_delta"]
al_vals = typed[typed["cq_type"] == "ALEATORIC"]["conf_delta"]
if len(al_vals) >= 2:
    stat, p = mannwhitneyu(ep_vals, al_vals, alternative="two-sided")
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"\nMann-Whitney U (EPISTEMIC vs ALEATORIC Δ): U={stat:.0f}, p={p:.4f} {sig}")
else:
    print(f"\nNote: Only {len(al_vals)} ALEATORIC CQ(s) — insufficient for significance test.")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Box / violin plot
order = ["EPISTEMIC", "ALEATORIC"]
data_plot = [typed[typed["cq_type"] == lbl]["conf_delta"].values for lbl in order]
bp = axes[0].boxplot(data_plot, labels=order, patch_artist=True,
                     medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], ["steelblue", "salmon"]):
    patch.set_facecolor(color)
axes[0].axhline(0, color="red", lw=1, linestyle="--")
axes[0].set_ylabel("Confidence delta (pp)")
axes[0].set_title("Δ Confidence by CQ Type")

# Scatter: preliminary_confidence vs conf_delta, coloured by cq_type
for lbl, color in [("EPISTEMIC", "steelblue"), ("ALEATORIC", "salmon")]:
    sub = typed[typed["cq_type"] == lbl]
    axes[1].scatter(sub["preliminary_confidence"], sub["conf_delta"],
                    c=color, label=lbl, alpha=0.7, s=50, edgecolors="none")
axes[1].axhline(0, color="gray", lw=1, linestyle="--")
axes[1].set_xlabel("Preliminary confidence")
axes[1].set_ylabel("Confidence delta (pp)")
axes[1].set_title("Δ Confidence vs Initial Confidence")
axes[1].legend()

fig.suptitle(f"CQ Type × Confidence Delta — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_cqtype_delta.png", dpi=150)
plt.show()
print("Saved: st_fig_cqtype_delta.png")

### 3.4 Confidence Delta by Category
Do some Microsoft product categories show larger confidence gains?

In [ ]:
cat_stats = (
    st_with_type
    .assign(conf_delta=lambda d: d["updated_confidence"] - d["preliminary_confidence"])
    .groupby("category")["conf_delta"]
    .agg(["mean", "median", "count"])
    .rename(columns={"mean": "Mean Δ", "median": "Median Δ", "count": "n"})
    .sort_values("Mean Δ", ascending=False)
)

print("Confidence delta by product category:")
display(cat_stats.round(1))

fig, ax = plt.subplots(figsize=(12, 5))
cats = cat_stats.index.tolist()
vals = cat_stats["Mean Δ"].values
colors = ["seagreen" if v >= 0 else "salmon" for v in vals]
ax.barh(cats, vals, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Mean Δ Confidence (pp)")
ax.set_title(f"Mean Confidence Delta by Category — {MODEL_ID}")
for i, (v, n) in enumerate(zip(vals, cat_stats["n"].values)):
    ax.text(v + (1 if v >= 0 else -1), i, f"n={n}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_category_delta.png", dpi=150)
plt.show()
print("Saved: st_fig_category_delta.png")

### 3.5 CQ Type by Category

In [ ]:
typed_cat = st_with_type[st_with_type["cq_type"].isin({"EPISTEMIC", "ALEATORIC"})]
cat_type = typed_cat.groupby(["category", "cq_type"]).size().unstack(fill_value=0)
cat_type["Total"] = cat_type.sum(axis=1)
for lbl in ["EPISTEMIC", "ALEATORIC"]:
    if lbl in cat_type.columns:
        cat_type[f"{lbl} %"] = (cat_type[lbl] / cat_type["Total"] * 100).round(1)

print("CQ type breakdown by category:")
display(cat_type)

---
## 4. Multi-Turn Analysis
*Three CQ rounds — 4 confidence checkpoints: Preliminary → After CQ1 → After CQ2 → Final*

### 4.1 Confidence Progression Across Checkpoints

In [ ]:
mt_ckpts = [
    ("Preliminary",  "preliminary_confidence"),
    ("After CQ1",    "confidence_1"),
    ("After CQ2",    "confidence_2"),
    ("Final",        "final_confidence"),
]

print("=== Multi-Turn Confidence Progression ===")
print(f"{'Checkpoint':<15} {'Mean':>7} {'Median':>8} {'Std':>7}")
print("-" * 40)
means = []
for label, col in mt_ckpts:
    m = mt[col].mean()
    med = mt[col].median()
    s = mt[col].std()
    means.append(m)
    print(f"  {label:<13} {m:>7.1f} {med:>8.1f} {s:>7.1f}")
print(f"\n  Total gain (Preliminary → Final): {means[-1] - means[0]:+.1f} pp")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Line: mean confidence at each checkpoint
labels = [l for l, _ in mt_ckpts]
axes[0].plot(labels, means, "o-", lw=2, ms=8, color="steelblue")
for i, (lbl, m) in enumerate(zip(labels, means)):
    axes[0].text(i, m + 0.5, f"{m:.1f}", ha="center", fontsize=9)
axes[0].set_ylabel("Mean Confidence")
axes[0].set_ylim(0, 100)
axes[0].set_title("Mean Confidence at Each Checkpoint")
axes[0].tick_params(axis="x", rotation=15)

# Individual case trajectories (thin grey) + mean (thick blue)
cols = [c for _, c in mt_ckpts]
for _, row in mt.iterrows():
    axes[1].plot(range(4), [row[c] for c in cols], color="gray", alpha=0.15, lw=0.8)
axes[1].plot(range(4), means, "o-", lw=2.5, ms=8, color="steelblue", label="Mean", zorder=5)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(labels, rotation=15)
axes[1].set_ylabel("Confidence")
axes[1].set_ylim(0, 100)
axes[1].set_title("Individual Case Trajectories")
axes[1].legend()

fig.suptitle(f"Multi-Turn Confidence Progression — {MODEL_ID} (n={len(mt)})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_confidence_arc.png", dpi=150)
plt.show()
print("Saved: mt_fig_confidence_arc.png")

### 4.2 CQ Type Distribution Per Turn

In [ ]:
q_col_mt = "clarifying_question" if "clarifying_question" in mt_labels.columns else "question"
mt_valid = mt_labels[mt_labels["label"].isin({"EPISTEMIC", "ALEATORIC"})].copy()

# mt_labels should have 'turn' column (judge writes id, turn, clarifying_question, label)
if "turn" not in mt_valid.columns:
    print("Warning: 'turn' column not found in mt_labels — building from results")
    long_rows = []
    for _, r in mt.iterrows():
        for t in range(1, 4):
            long_rows.append({"id": r["id"], "turn": t, q_col_mt: r[f"cq_{t}"]})
    long_df = pd.DataFrame(long_rows)
    mt_valid = mt_valid.merge(long_df, on=["id", q_col_mt], how="left")

print(f"Multi-turn CQs classified: {len(mt_valid)} (expected {len(mt)*3})")
print()

# Overall distribution
vc_mt = mt_valid["label"].value_counts()
for lbl in ["EPISTEMIC", "ALEATORIC"]:
    n = vc_mt.get(lbl, 0)
    print(f"  Overall {lbl}: {n} ({100*n/len(mt_valid):.1f}%)")

print()
print("By turn:")
turn_dist = mt_valid.groupby(["turn", "label"]).size().unstack(fill_value=0)
display(turn_dist)

# Plot CQ type per turn
turns = [1, 2, 3]
ep_pct = []
al_pct = []
for t in turns:
    sub = mt_valid[mt_valid["turn"] == t]
    total = len(sub)
    ep_pct.append(100 * sub[sub["label"]=="EPISTEMIC"].shape[0] / total if total else 0)
    al_pct.append(100 * sub[sub["label"]=="ALEATORIC"].shape[0] / total if total else 0)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(turns))
ax.bar(x - 0.18, ep_pct, 0.35, label="EPISTEMIC", color="steelblue")
ax.bar(x + 0.18, al_pct, 0.35, label="ALEATORIC", color="salmon")
ax.set_xticks(x)
ax.set_xticklabels([f"CQ{t}" for t in turns])
ax.set_ylabel("% of CQs")
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(f"CQ Type by Turn — {MODEL_ID}")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_cq_type_per_turn.png", dpi=150)
plt.show()
print("Saved: mt_fig_cq_type_per_turn.png")

### 4.3 Confidence Delta × CQ Type Per Turn  ← *Core Research Result*
Does asking an EPISTEMIC CQ at turn T lead to a larger confidence gain at turn T+1?

In [ ]:
from scipy.stats import mannwhitneyu

# Build per-turn delta table
conf_cols = [
    (1, "preliminary_confidence", "confidence_1"),
    (2, "confidence_1",           "confidence_2"),
    (3, "confidence_2",           "final_confidence"),
]

rows = []
for _, r in mt.iterrows():
    for turn_idx, c_before, c_after in conf_cols:
        cq_text = r.get(f"cq_{turn_idx}", "")
        label_rows = mt_valid[(mt_valid["id"] == r["id"]) & (mt_valid["turn"] == turn_idx)]
        lbl = label_rows["label"].iloc[0] if len(label_rows) else None
        rows.append({
            "id": r["id"],
            "turn": turn_idx,
            "cq_type": lbl,
            "conf_before": r[c_before],
            "conf_after":  r[c_after],
            "conf_delta":  r[c_after] - r[c_before],
        })

turn_df = pd.DataFrame(rows)
typed_turn = turn_df[turn_df["cq_type"].isin({"EPISTEMIC", "ALEATORIC"})]

print("=== Confidence Delta by Turn × CQ Type ===")
print()
for t in [1, 2, 3]:
    sub = typed_turn[typed_turn["turn"] == t]
    ep = sub[sub["cq_type"]=="EPISTEMIC"]["conf_delta"]
    al = sub[sub["cq_type"]=="ALEATORIC"]["conf_delta"]
    print(f"  Turn {t}:  EPISTEMIC mean={ep.mean():+.1f} (n={len(ep)})  "
          f"| ALEATORIC mean={al.mean():+.1f} (n={len(al)})")
    if len(al) >= 2 and len(ep) >= 2:
        stat, p = mannwhitneyu(ep, al, alternative="two-sided")
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
        print(f"           Mann-Whitney: p={p:.4f} {sig}")
    else:
        print(f"           (insufficient ALEATORIC cases for test)")

print()
print("Pooled across all turns:")
ep_all = typed_turn[typed_turn["cq_type"]=="EPISTEMIC"]["conf_delta"]
al_all = typed_turn[typed_turn["cq_type"]=="ALEATORIC"]["conf_delta"]
print(f"  EPISTEMIC: mean={ep_all.mean():+.1f}, median={ep_all.median():+.1f}, n={len(ep_all)}")
print(f"  ALEATORIC: mean={al_all.mean():+.1f}, median={al_all.median():+.1f}, n={len(al_all)}")
if len(al_all) >= 2:
    stat, p = mannwhitneyu(ep_all, al_all, alternative="two-sided")
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"  Mann-Whitney (pooled): p={p:.4f} {sig}")

# Visualise per-turn delta by type
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, t in zip(axes, [1, 2, 3]):
    sub = typed_turn[typed_turn["turn"] == t]
    data = [sub[sub["cq_type"]==lbl]["conf_delta"].values for lbl in ["EPISTEMIC", "ALEATORIC"]]
    bp = ax.boxplot(data, labels=["EPISTEMIC", "ALEATORIC"], patch_artist=True,
                    medianprops=dict(color="black", lw=2))
    for patch, color in zip(bp["boxes"], ["steelblue", "salmon"]):
        patch.set_facecolor(color)
    ax.axhline(0, color="red", lw=1, linestyle="--")
    ax.set_title(f"CQ{t}")
    if t == 1:
        ax.set_ylabel("Confidence delta (pp)")

fig.suptitle(f"Conf Delta by CQ Type per Turn — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_cqtype_delta_per_turn.png", dpi=150)
plt.show()
print("Saved: mt_fig_cqtype_delta_per_turn.png")

### 4.4 Cross-Turn CQ Type Consistency
Does the model consistently choose EPISTEMIC vs ALEATORIC across turns for the same case?

In [ ]:
# Pivot to wide format: one row per case, columns cq1_type, cq2_type, cq3_type
wide = mt_valid.pivot_table(index="id", columns="turn", values="label", aggfunc="first")
wide.columns = [f"cq{t}_type" for t in wide.columns]
wide = wide.reset_index()

# Cases where all 3 CQs are the same type
if "cq1_type" in wide.columns and "cq2_type" in wide.columns and "cq3_type" in wide.columns:
    all_same = wide.apply(lambda r: r["cq1_type"]==r["cq2_type"]==r["cq3_type"], axis=1)
    print(f"Cases with same CQ type all 3 turns: {all_same.sum()}/{len(wide)} ({100*all_same.mean():.1f}%)")
    print()
    # Pattern distribution
    if all(c in wide.columns for c in ["cq1_type", "cq2_type", "cq3_type"]):
        patterns = wide[["cq1_type","cq2_type","cq3_type"]].apply(
            lambda r: "→".join([str(r[c])[0] for c in ["cq1_type","cq2_type","cq3_type"]]), axis=1
        ).value_counts()
        print("CQ type sequence patterns (E=EPISTEMIC, A=ALEATORIC):")
        for pat, n in patterns.items():
            print(f"  {pat.replace('E', 'E').replace('A', 'A')}: {n} cases")
else:
    print("Columns:", wide.columns.tolist())

### 4.5 CQ Lexical Diversity (Jaccard)
Are the CQs across turns asking different things?

In [ ]:
print("Mean token Jaccard similarity between CQ pairs (lower = more diverse):")
for ca, cb in [("cq_1","cq_2"), ("cq_2","cq_3"), ("cq_1","cq_3")]:
    sims = mt.apply(lambda r: jaccard(r[ca], r[cb]), axis=1)
    print(f"  {ca} vs {cb}: {sims.mean():.3f}  (median {sims.median():.3f})")

### 4.6 Simulator Response Quality
How often does the simulator give an informative answer vs. a non-answer?

In [ ]:
print("Fraction of simulator responses containing uncertainty indicators:")
hedges = ["not sure", "don't know", "haven't checked", "not available", "unsure", "i don't"]
for t in [1, 2, 3]:
    responses = mt[f"user_response_{t}"].fillna("").str.lower()
    n_hedge = responses.apply(lambda x: any(h in x for h in hedges)).sum()
    print(f"  CQ{t}: {n_hedge}/{len(mt)} ({100*n_hedge/len(mt):.1f}%) non-committal responses")

---
## 5. Cross-Experiment Comparison
*Single-turn vs Multi-turn on the same 100 cases*

In [ ]:
st_delta = st["updated_confidence"] - st["preliminary_confidence"]
mt_final_delta = mt["final_confidence"] - mt["preliminary_confidence"]

print("=== Confidence Summary ===")
print(f"{'':30} {'Single-turn':>14} {'Multi-turn':>12}")
print("-" * 58)
print(f"  {'Preliminary mean':28} {st['preliminary_confidence'].mean():>14.1f} {mt['preliminary_confidence'].mean():>12.1f}")
print(f"  {'Post-CQ1 mean':28} {st['updated_confidence'].mean():>14.1f} {mt['confidence_1'].mean():>12.1f}")
print(f"  {'Final mean':28} {'(same as CQ1)':>14} {mt['final_confidence'].mean():>12.1f}")
print(f"  {'Total Δ mean':28} {st_delta.mean():>+14.1f} {mt_final_delta.mean():>+12.1f}")
print()

# EPISTEMIC % comparison
st_ep = 100 * vc.get("EPISTEMIC", 0) / len(st_valid) if len(st_valid) else 0
mt_ep = 100 * vc_mt.get("EPISTEMIC", 0) / len(mt_valid) if len(mt_valid) else 0
print(f"  {'EPISTEMIC CQ %':28} {st_ep:>14.1f}% {mt_ep:>12.1f}%")

# Plot: confidence progression comparison
fig, ax = plt.subplots(figsize=FIGSIZE)

# Single-turn: prelim → updated
ax.plot([0, 1], [st['preliminary_confidence'].mean(), st['updated_confidence'].mean()],
        "o-", lw=2.5, ms=8, color="steelblue", label="Single-turn")

# Multi-turn: 4 checkpoints
mt_means = [mt[c].mean() for _, c in mt_ckpts]
ax.plot([0, 1, 2, 3], mt_means,
        "s-", lw=2.5, ms=8, color="seagreen", label="Multi-turn")

ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(["Preliminary", "After CQ1", "After CQ2", "Final"])
ax.set_ylabel("Mean Confidence")
ax.set_ylim(0, 100)
ax.set_title(f"Confidence Progression: Single-Turn vs Multi-Turn — {MODEL_ID}")
ax.legend()
ax.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.savefig(OUTPUTS / "fig_comparison.png", dpi=150)
plt.show()
print("Saved: fig_comparison.png")

---
## 6. Export Summary CSV

In [ ]:
# ── Single-turn summary ────────────────────────────────────────────────────────
st_export = st[["id", "category", "preliminary_confidence", "updated_confidence",
                "clarifying_question"]].copy()
st_export["conf_delta"] = st_export["updated_confidence"] - st_export["preliminary_confidence"]

if len(st_valid) > 0:
    cq_type_map = st_valid.set_index(q_col_st)["label"].to_dict()
    st_export["cq_type"] = st_export["clarifying_question"].map(cq_type_map)

st_out = OUTPUTS / "st_analysis_summary.csv"
st_export.to_csv(st_out, index=False)
print(f"Single-turn summary: {st_out.name}  ({len(st_export)} rows)")

# ── Multi-turn summary ─────────────────────────────────────────────────────────
mt_export = mt[["id", "category",
                "preliminary_confidence", "confidence_1", "confidence_2", "final_confidence",
                "cq_1", "cq_2", "cq_3"]].copy()
mt_export["total_delta"] = mt_export["final_confidence"] - mt_export["preliminary_confidence"]

# Attach CQ type per turn
for t in [1, 2, 3]:
    turn_types = mt_valid[mt_valid["turn"]==t][["id","label"]].rename(columns={"label": f"cq{t}_type"})
    mt_export = mt_export.merge(turn_types, on="id", how="left")

mt_out = OUTPUTS / "mt_analysis_summary.csv"
mt_export.to_csv(mt_out, index=False)
print(f"Multi-turn summary:  {mt_out.name}  ({len(mt_export)} rows)")
display(mt_export.head(3))